In [20]:
import pickle  # to save the fitted vectorizer and model.
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split


In [25]:
PROJECT_ROOT  = Path("..")

DATA_DIR = PROJECT_ROOT / "data/raw/"
MODEL_DIR = PROJECT_ROOT / "models/"


DATA_PATH = DATA_DIR / "green_tripdata_2026-01.parquet"
MODEL_PATH = MODEL_DIR / "baseline.pkl"

MODEL_DIR.mkdir(parents=True, exist_ok=True)



if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Download the NYC TLC Green Taxi parquet file first."
    )

print("Dataset:", DATA_PATH)
print("Model output:", MODEL_PATH)


Dataset: ../data/raw/green_tripdata_2026-01.parquet
Model output: ../models/baseline.pkl


In [22]:

df = pd.read_parquet(path=DATA_PATH)

df.head()


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,1,2026-01-01 00:27:58,2026-01-01 00:55:16,N,1.0,65,233,2.0,6.20,31.7,...,1.5,7.5,0.0,NaN,1.0,45.20,1.0,1.0,2.75,0.75
1,2,2026-01-01 00:44:33,2026-01-01 01:32:56,N,5.0,66,188,5.0,5.36,50.0,...,0.0,10.2,0.0,NaN,1.0,61.20,1.0,2.0,0.00,0.00
2,1,2026-01-01 00:23:45,2026-01-01 00:45:03,N,1.0,65,179,4.0,10.60,41.5,...,1.5,2.0,0.0,NaN,1.0,46.00,1.0,1.0,0.00,0.00
3,1,2026-01-01 00:44:33,2026-01-01 01:00:45,N,1.0,42,141,1.0,4.20,19.8,...,1.5,0.0,0.0,NaN,1.0,25.05,2.0,1.0,2.75,0.00
4,2,2026-01-01 00:46:04,2026-01-01 01:04:40,N,1.0,95,82,1.0,2.76,19.1,...,0.5,0.0,0.0,NaN,1.0,21.60,2.0,1.0,0.00,0.00


In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40272 entries, 0 to 40271
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   VendorID               40272 non-null  int32         
 1   lpep_pickup_datetime   40272 non-null  datetime64[us]
 2   lpep_dropoff_datetime  40272 non-null  datetime64[us]
 3   store_and_fwd_flag     34858 non-null  str           
 4   RatecodeID             34858 non-null  float64       
 5   PULocationID           40272 non-null  int32         
 6   DOLocationID           40272 non-null  int32         
 7   passenger_count        34858 non-null  float64       
 8   trip_distance          40272 non-null  float64       
 9   fare_amount            40272 non-null  float64       
 10  extra                  40272 non-null  float64       
 11  mta_tax                40272 non-null  float64       
 12  tip_amount             40272 non-null  float64       
 13  tolls_amount

In [26]:
df['duration'] = (df.lpep_dropoff_datetime - df.lpep_pickup_datetime).dt.total_seconds()/ 60


In [27]:
df = df[ (df.duration >= 1 ) & (df.duration <= 60)].copy()

original_rows = len(df)

df = df[ (df.duration >= 1 ) & (df.duration <= 60)].copy()

remaining_rows = len(df)

print(f"Rows before cleaning: {original_rows:,}")
print(f"Rows after cleaning:  {remaining_rows:,}")
print(f"Rows removed:         {original_rows - remaining_rows:,}")

Rows before cleaning: 38,088
Rows after cleaning:  38,088
Rows removed:         0


In [28]:
columns_needed = [
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "duration",
]

df[columns_needed].isna().sum()

PULocationID     0
DOLocationID     0
trip_distance    0
duration         0
dtype: int64

In [18]:
df["duration"].describe()

count    38088.000000
mean        15.581492
std         10.310362
min          1.000000
25%          8.433333
50%         12.950000
75%         19.633333
max         60.000000
Name: duration, dtype: float64

In [29]:
df = df.dropna(
    subset=[
        "PULocationID",
        "DOLocationID",
        "trip_distance",
        "duration",
    ]
).copy()

print(f"Rows remaining: {len(df):,}")

Rows remaining: 38,088


In [30]:
# categorical data -> each ID for zone -> we need to covert them to numerical feature
# we make route for trip as string -> encoding them

df["PU_DO"] =  df["PULocationID"].astype(str)  + "_" +  df["DOLocationID"].astype(str)

In [32]:
df[
    [
        "PULocationID",
        "DOLocationID",
        "PU_DO",
        "trip_distance",
        "duration",
    ]
].head()

,PULocationID,DOLocationID,PU_DO,trip_distance,duration
0,65,233,65_233,6.20,27.300000
1,66,188,66_188,5.36,48.383333
2,65,179,65_179,10.60,21.300000
3,42,141,42_141,4.20,16.200000
4,95,82,95_82,2.76,18.600000


In [34]:
categorical_features = ["PU_DO"]
numerical_features = ["trip_distance"]

features = categorical_features + numerical_features

target = "duration"

print("Features:", features)
print("Target:", target)

Features: ['PU_DO', 'trip_distance']
Target: duration


In [35]:
df_train, df_val = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
)

print(f"Training rows:   {len(df_train):,}")
print(f"Validation rows: {len(df_val):,}")

Training rows:   30,470
Validation rows: 7,618


### Use Dict Vectorizer -> convert dictionaries into feature vectors 
### (one hot encoding )

In [ ]:

train_dicts = df_train[features].to_dict(orient="records") #Mlist of dicts
val_dicts = df_val[features].to_dict(orient="records")


dv = DictVectorizer()

X_train = dv.fit_transform(train_dicts)

X_val = dv.transform(val_dicts)


<class 'list'>


In [40]:
print("Training matrix shape:", X_train.shape)
print("Validation matrix shape:", X_val.shape)

Training matrix shape: (30470, 4499)
Validation matrix shape: (7618, 4499)


In [41]:
feature_names = dv.get_feature_names_out()

print("Number of generated features:", len(feature_names))
print(feature_names[:20])

Number of generated features: 4499
['PU_DO=100_116' 'PU_DO=100_18' 'PU_DO=100_247' 'PU_DO=100_35'
 'PU_DO=100_74' 'PU_DO=100_86' 'PU_DO=101_205' 'PU_DO=101_95'
 'PU_DO=101_98' 'PU_DO=102_177' 'PU_DO=102_225' 'PU_DO=102_82'
 'PU_DO=102_9' 'PU_DO=102_95' 'PU_DO=106_106' 'PU_DO=106_112'
 'PU_DO=106_14' 'PU_DO=106_189' 'PU_DO=106_198' 'PU_DO=106_256']


In [42]:
y_train = df_train[target].to_numpy()
y_val = df_val[target].to_numpy()

print(y_train.shape)
print(y_val.shape)

(30470,)
(7618,)


In [43]:
model = LinearRegression()

model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](4499,)","[ 4.02, 7.91, 19.55,...,-19.54,-19.19, -0. ]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,26.44
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,4499


In [45]:
y_pred = model.predict(X_val)

y_pred[:10]

array([ 7.75093224, 16.53437579, 45.65278693, 15.17172202, 12.3511975 ,
       26.61495653, 26.61496457,  8.47517986, 18.29521902, 17.07189228])

In [47]:
mae = mean_absolute_error(
    y_val,
    y_pred,
)

print(f"Validation MAE: {mae:.4f} minutes")

mse = mean_squared_error(
    y_val,
    y_pred,
)

rmse = np.sqrt(mse)

print(f"Validation RMSE: {rmse:.4f} minutes")


Validation MAE: 4.7983 minutes
Validation RMSE: 7.2011 minutes


In [48]:
sample = {
    "PU_DO": df_val.iloc[0]["PU_DO"],
    "trip_distance": df_val.iloc[0]["trip_distance"],
}

sample_vector = dv.transform([sample])

sample_prediction = model.predict(sample_vector)[0]

print(f"Predicted duration: {sample_prediction:.2f} minutes")
print(f"Actual duration:    {df_val.iloc[0]['duration']:.2f} minutes")

Predicted duration: 7.75 minutes
Actual duration:    5.52 minutes


# Save the model artifact

In [50]:
artifact = {
    "vectorizer": dv,
    "model": model,
    "features": features,
    "target": target,
    "metrics": {
        "mae": float(mae),
        "rmse": float(rmse),
    },
}

with MODEL_PATH.open("wb") as f:
    pickle.dump(artifact, f)

print(f"Saved model artifact to: {MODEL_PATH}")

Saved model artifact to: ../models/baseline.pkl


In [51]:
assert MODEL_PATH.exists()

print(
    f"Artifact size: "
    f"{MODEL_PATH.stat().st_size / 1024:.2f} KB"
)

Artifact size: 136.36 KB


In [52]:
with MODEL_PATH.open("rb") as f:
    loaded_artifact = pickle.load(f)

loaded_dv = loaded_artifact["vectorizer"]
loaded_model = loaded_artifact["model"]

In [53]:
loaded_sample_vector = loaded_dv.transform([sample])

loaded_prediction = loaded_model.predict(
    loaded_sample_vector
)[0]

print(
    f"Original model prediction: "
    f"{sample_prediction:.6f}"
)

print(
    f"Reloaded model prediction: "
    f"{loaded_prediction:.6f}"
)

Original model prediction: 7.750932
Reloaded model prediction: 7.750932
